# Historical Daily Bars Pipeline

This notebook is the executable extraction and validation stage of the Equity Data Platform. Kestra can run every cell from top to bottom without manual input.

## 1. Import pipeline dependencies

The following cell imports the standard-library utilities, Alpaca request models, pandas, and the project's configuration and client factories used by later cells.

In [1]:
import os
from datetime import date, datetime, time, timedelta

import pandas as pd
from alpaca.data.enums import DataFeed
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

from equity_data_platform.clients import create_historical_client
from equity_data_platform.config import get_settings

## 2. Initialize configuration and the Alpaca client

This cell loads API credentials from environment settings, reads the configured stock symbols and IEX feed, and creates the authenticated historical-data client. It never prints secret values.

In [2]:
settings = get_settings()
client = create_historical_client(settings)

symbols = list(settings.symbols)
feed = DataFeed(settings.feed)
print("Configured symbols:", symbols)
print("Market-data feed:", feed.value)

Configured symbols: ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'META']
Market-data feed: iex


## 3. Build the historical-data request

This cell reads an inclusive date range from `ALPACA_START_DATE` and `ALPACA_END_DATE`. Kestra can inject these environment variables for scheduled runs or backfills, while the defaults keep the notebook runnable locally.

In [3]:
start_date = date.fromisoformat(os.getenv("ALPACA_START_DATE", "2026-10-01"))
end_date = date.fromisoformat(os.getenv("ALPACA_END_DATE", "2026-10-16"))

if start_date > end_date:
    raise ValueError("ALPACA_START_DATE must not be later than ALPACA_END_DATE")

start_at = datetime.combine(start_date, time.min)
end_at = datetime.combine(end_date + timedelta(days=1), time.min)

request = StockBarsRequest(
    symbol_or_symbols=symbols,
    timeframe=TimeFrame.Day,
    start=start_at,
    end=end_at,
    feed=feed,
)

## 4. Extract and normalize daily bars

This cell sends one multi-symbol request to Alpaca and converts the SDK response into a flat DataFrame. Resetting the index turns `symbol` and `timestamp` into ordinary columns suitable for validation and future serialization.

In [4]:
response = client.get_stock_bars(request)
bars_df = response.df.reset_index()

print(f"Extracted {len(bars_df):,} bar records.")

Extracted 60 bar records.


## 5. Define the bar-data quality contract

This cell defines the schema and business rules required before records can continue downstream. The function checks completeness, uniqueness, OHLC consistency, non-negative activity, VWAP bounds, and coverage of every requested symbol. It uses explicit exceptions so Kestra receives a failed task status when the contract is violated.

In [5]:
REQUIRED_BAR_COLUMNS = (
    "symbol",
    "timestamp",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "trade_count",
    "vwap",
)


def validate_bars(df: pd.DataFrame, expected_symbols: list[str]) -> None:
    missing_columns = sorted(set(REQUIRED_BAR_COLUMNS) - set(df.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    if df.empty:
        raise ValueError("Alpaca returned no bar records")

    price_columns = ["open", "high", "low", "close"]
    invalid_ohlc = (df["high"] != df[price_columns].max(axis=1)) | (
        df["low"] != df[price_columns].min(axis=1)
    )

    failure_counts = {
        "rows with missing required values": int(
            df[list(REQUIRED_BAR_COLUMNS)].isna().any(axis=1).sum()
        ),
        "duplicate (symbol, timestamp) rows": int(
            df.duplicated(subset=["symbol", "timestamp"], keep=False).sum()
        ),
        "rows with invalid OHLC bounds": int(invalid_ohlc.sum()),
        "rows with negative volume": int((df["volume"] < 0).sum()),
        "rows with negative trade count": int((df["trade_count"] < 0).sum()),
        "rows with VWAP outside low/high": int(
            ((df["vwap"] < df["low"]) | (df["vwap"] > df["high"])).sum()
        ),
    }

    returned_symbols = set(df["symbol"].dropna().unique())
    missing_symbols = sorted(set(expected_symbols) - returned_symbols)
    failures = {name: count for name, count in failure_counts.items() if count}

    if missing_symbols:
        failures["requested symbols with no returned bars"] = missing_symbols

    if failures:
        details = "; ".join(f"{name}: {value}" for name, value in failures.items())
        raise ValueError(f"Bar data validation failed: {details}")

## 6. Enforce the quality contract

This cell applies the validation function to the extracted batch. Successful execution means the dataset satisfies the current requirements; any exception stops the task before invalid data reaches the next pipeline stage.

In [6]:
validate_bars(bars_df, symbols)
print("Bar-data validation passed.")

Bar-data validation passed.


## 7. Emit an operational summary

This cell writes concise batch information and sample records to the task log so an operator can verify the completed extraction. Timestamps are shown exactly as returned by Alpaca.

In [7]:
rows_per_symbol = bars_df.groupby("symbol").size().rename("row_count")
preview_df = bars_df.head()

print(f"Requested date range: {start_date} through {end_date}")
print("Rows per symbol:")
print(rows_per_symbol.to_string())
print("Sample records:")
print(preview_df.to_string(index=False))

Requested date range: 2026-10-01 through 2026-10-16
Rows per symbol:
symbol
AAPL    12
AMZN    12
META    12
MSFT    12
NVDA    12
Sample records:
symbol                 timestamp    open    high     low  close    volume  trade_count       vwap
  AAPL 2026-10-01 04:00:00+00:00 329.550 332.470 325.810 330.44  988015.0      19383.0 329.366790
  AAPL 2026-10-02 04:00:00+00:00 333.160 334.525 330.660 333.75  841636.0      18894.0 332.919606
  AAPL 2026-10-05 04:00:00+00:00 332.820 336.190 331.660 333.11 1105512.0      21287.0 333.590277
  AAPL 2026-10-06 04:00:00+00:00 332.305 334.365 330.635 333.72  728478.0      17717.0 332.961592
  AAPL 2026-10-07 04:00:00+00:00 337.015 338.600 332.820 336.58 1211339.0      22394.0 335.990341


## Conversion to a Python task

The notebook intentionally uses ordinary Python without notebook magics or interactive display functions. Its cells can therefore be moved into a `.py` entry point with minimal changes when the Kestra workflow is finalized. The GCS write stage will be added after cloud storage integration.